In [1]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime

from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

EMBED_PATH = ROOT / "data/embeddings/base_embeds.pt"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = ROOT / "data/experiments" / EMBED_NAME / "ellipsoid"
RESULTS_DIR = ROOT / "data/results" / EMBED_NAME

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

In [3]:
embeds = torch.load(EMBED_PATH, weights_only=False)
cls_tokens = embeds["cls_tokens"]

In [4]:
conn = sqlite3.connect(ROOT / "data/sql/metadata.db")

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [5]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid import EllipsoidFitter, EllipsoidCover, CandidateCleaner, EllipsoidEvaluator

fitter = EllipsoidFitter(support_points=5, reg=1e-4)
cleaner = CandidateCleaner(min_points=1, fitter=fitter)

cover = EllipsoidCover(fitter=fitter, cleaner=cleaner)

eval = EllipsoidEvaluator()

In [ ]:
time = datetime.now().strftime("%y-%m-%d_%H-%M-%S")
aurocs = {}

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category / time
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[~train_mask]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    ellipsoids, ellipsoids_df = cover.run(cat_emb, output_dir=outputs_dir)
    
    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[~train_mask]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    overlaps_df, num_overlaps = eval.overlap(cat_emb, ellipsoids)
    overlaps_df.to_csv(outputs_dir / f"overlaps.csv", index=False)

    good_any, good_counts = eval.inside_any_count(good_test_emb, ellipsoids)
    defect_any, defect_counts = eval.inside_any_count(defect_test_emb, ellipsoids)

    results_df, metrics = eval.evaluate_detection(good_test_emb, defect_test_emb, ellipsoids)
    results_df.to_csv(outputs_dir / f"results.csv", index=False)

    diagnostics = eval.bucket_diagnostics(results_df)

    aurocs[category] = metrics["auroc"]

    metadata = {
        "category": category,
        "K_frac": 0.05,
        "start_growth": 1.2,
        "min_growth": 1,
        "reg": 1e-4,
        "growth_type": "variance_scaled",
        "cleaner": "shared_axis",
        "n_ellipsoids": len(ellipsoids),
        "auroc": metrics["auroc"],
        "good_inside": int(good_any.sum()),
        "defect_inside": int(defect_any.sum())
    }

    with open(outputs_dir / f"metadata.json", "w") as f:
        json.dump(metadata, f)

aurocs_df = pd.DataFrame(aurocs.items(), columns=["Category", "AUROC"])

Running bottle
Running cable
Running capsule
Running carpet
Running grid
Running hazelnut
Running leather
Running metal_nut
Running pill
Running screw
Running tile
Running toothbrush
Running transistor
Running wood
Running zipper


In [ ]:
df_roc_stats = pd.DataFrame({
    "mean": aurocs_df["AUROC"].mean(),
    "median": aurocs_df["AUROC"].median(),
    "std": aurocs_df["AUROC"].std(),
    "min_cat":  aurocs_df["Category"][aurocs_df["AUROC"].idxmin()],
    "min": aurocs_df["AUROC"].min(),
    "max_cat": aurocs_df["Category"][aurocs_df["AUROC"].idxmax()],
    "max": aurocs_df["AUROC"].max()
}, index=[0]).round(3)

df_roc_stats.to_csv(RESULTS_DIR / "ellipsoid_auroc_stats.csv", index=False)

aurocs_df = aurocs_df.round(3)
aurocs_df.to_csv(RESULTS_DIR /"ellipsoid_aurocs.csv", index=False)

df_roc_stats

,mean,median,std,min_cat,min,max_cat,max
0,0.901,0.934,0.099,transistor,0.688,bottle,1.0
